#Name : Saksham Sigdel
##Std ID: 2418223

#Step 1 – Mount Google Drive





In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Step 2 – Import libraries and define helper functions

---



In [22]:
# Import necessary libraries
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Download required NLTK data (run once)
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

# ---------- Helper functions for text cleaning ----------

def remove_urls(text):
    """Remove URLs from text using regex"""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def remove_emoji(string):
    """Remove emojis from text using Unicode ranges"""
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                               u"\U00002702-\U000027B0"
                               u"\U000024C2-\U0001F251"
                               "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r' ', string)

def removeunwanted_characters(document):
    """Remove mentions, hashtags, punctuation, and extra spaces"""
    # Remove user mentions (e.g., @username)
    document = re.sub("@[A-Za-z0-9_]+", " ", document)
    # Remove hashtags (e.g., #topic)
    document = re.sub("#[A-Za-z0-9_]+", "", document)
    # Remove punctuation and any non-alphanumeric character (keep letters, numbers, spaces)
    document = re.sub("[^0-9A-Za-z ]", "", document)
    # Remove emojis (call the emoji remover)
    document = remove_emoji(document)
    # Replace double spaces with single space
    document = document.replace('  ', '')
    return document.strip()

def lemmatization(token_text):
    """Lemmatize each token (convert to base form, e.g., 'running' -> 'run')"""
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(token, pos='v') for token in token_text]

def stemming(token_text):
    """Stem each token (cut off suffixes, e.g., 'running' -> 'run')"""
    stemmer = PorterStemmer()
    return [stemmer.stem(word) for word in token_text]

# Set up stopwords (common words like 'the', 'and' that add little meaning)
stop_words = set(stopwords.words('english'))
# Add custom stopwords that appear often in tweets after cleaning
stop_words.update(['rt', 'https', 'com'])

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


#Step 3 – Complete the text_cleaning_pipeline function

In [31]:
def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    Complete text cleaning pipeline:
    1. Lowercase the text
    2. Remove "rt " (retweet indicator)
    3. Remove URLs
    4. Remove emojis
    5. Remove mentions, hashtags, punctuation, special characters
    6. Tokenize
    7. Remove stopwords
    8. Lemmatize or stem
    """
    # Step 1: Lowercase
    data = dataset.lower()

    # Step 2: Remove "rt " word (retweet indicator) to prevent gluing
    data = re.sub(r'\brt\b', '', data)

    # Step 3: Remove URLs
    data = remove_urls(data)

    # Step 4: Remove emojis
    data = remove_emoji(data)

    # Step 5: Remove unwanted characters (mentions, hashtags, punctuation)
    data = removeunwanted_characters(data)

    # Step 6: Tokenize
    tokens = data.split()

    # Step 7: Remove stopwords
    tokens = [token for token in tokens if token not in stop_words]

    # Step 8: Lemmatize or stem
    if rule == "lemmatize":
        tokens = lemmatization(tokens)
    elif rule == "stem":
        tokens = stemming(tokens)
    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)

#Step 4 – Load the labeled dataset

In [32]:
# Path to your labeled dataset (contains 'text' and 'label' columns)
file_path = '/content/drive/MyDrive/Ai/trum_tweet_sentiment_analysis.csv'
df = pd.read_csv(file_path)

# Display column names to understand structure
print("Columns in dataset:", df.columns.tolist())

# Rename sentiment column to 'label' if it's named differently
if 'Sentiment' in df.columns:
    df.rename(columns={'Sentiment': 'label'}, inplace=True)
elif 'sentiment' in df.columns:
    df.rename(columns={'sentiment': 'label'}, inplace=True)

# Ensure the text column is named 'text' (could be 'tweet' or 'content')
if 'text' not in df.columns and 'tweet' in df.columns:
    df.rename(columns={'tweet': 'text'}, inplace=True)
elif 'text' not in df.columns and 'content' in df.columns:
    df.rename(columns={'content': 'text'}, inplace=True)

# Verify that required columns exist
assert 'text' in df.columns, "Error: 'text' column not found in dataset"
assert 'label' in df.columns, "Error: 'label' column not found in dataset"

print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows of text and label:")
print(df[['text', 'label']].head())

Columns in dataset: ['text', 'Sentiment']
Dataset shape: (1850123, 2)

First 5 rows of text and label:
                                                text  label
0  RT @JohnLeguizamo: #trump not draining swamp b...      0
1  ICYMI: Hackers Rig FM Radio Stations To Play A...      0
2  Trump protests: LGBTQ rally in New York https:...      1
3  "Hi I'm Piers Morgan. David Beckham is awful b...      0
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...      0


#Step 5 – Clean the text column

In [33]:
# Apply the cleaning pipeline to each tweet (using lemmatization)
df['cleaned_text'] = df['text'].apply(lambda x: text_cleaning_pipeline(x, rule='lemmatize'))

# Display a sample comparison
print("Original tweet (first row):")
print(df['text'].iloc[0][:200])
print("\nCleaned version (first row):")
print(df['cleaned_text'].iloc[0][:200])

Original tweet (first row):
RT @JohnLeguizamo: #trump not draining swamp but our taxpayer dollars on his trips to advertise his properties! @realDonaldTrump https://t.co/gFBvUkMX9z

Cleaned version (first row):
drain swamp taxpayer dollars trip advertise properties


#Step 6 – Train‑test split

In [34]:
from sklearn.model_selection import train_test_split

# Features (cleaned text) and target (sentiment labels)
X = df['cleaned_text']
y = df['label']

# Split: 80% training, 20% testing, random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 1480098
Testing samples: 370025


#Step 7 – TF‑IDF vectorization

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert text to numerical feature vectors using TF‑IDF
# max_features limits vocabulary size for efficiency
vectorizer = TfidfVectorizer(max_features=5000)

# Fit on training data and transform both training and test sets
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF training matrix shape: {X_train_tfidf.shape}")
print(f"TF-IDF test matrix shape: {X_test_tfidf.shape}")

TF-IDF training matrix shape: (1480098, 5000)
TF-IDF test matrix shape: (370025, 5000)


#Step 8 – Train Logistic Regression and print classification report

In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Initialize logistic regression model
model = LogisticRegression(max_iter=1000, random_state=42)

# Train the model
model.fit(X_train_tfidf, y_train)

# Predict on test set
y_pred = model.predict(X_test_tfidf)

# Print performance metrics
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.96      0.94    248563
           1       0.90      0.86      0.88    121462

    accuracy                           0.92    370025
   macro avg       0.92      0.91      0.91    370025
weighted avg       0.92      0.92      0.92    370025



#Step 9 – (Optional) Show sample predictions

In [37]:
import random

# Pick 5 random test samples
indices = random.sample(range(len(X_test)), 5)

for idx in indices:
    print(f"Tweet snippet: {X_test.iloc[idx][:100]}...")
    print(f"Predicted sentiment: {y_pred[idx]} | Actual sentiment: {y_test.iloc[idx]}")
    print()

Tweet snippet: embarrassment president trumpvia...
Predicted sentiment: 0 | Actual sentiment: 0

Tweet snippet: racists think get bold without consequence trump...
Predicted sentiment: 0 | Actual sentiment: 0

Tweet snippet: trump let flynn get away humiliate emasculate pence russians must potus even figure...
Predicted sentiment: 0 | Actual sentiment: 0

Tweet snippet: break maine gop senator susan collins say vote trump epa nominee scott pruittvia...
Predicted sentiment: 0 | Actual sentiment: 0

Tweet snippet: photosin 2008 putin approve donald trump jr keynote speaker russian real estate conference moscow...
Predicted sentiment: 1 | Actual sentiment: 1

